# 조성 기술자 실습

**Composition Descriptor · Magpie · 원소 특성 통계**

조성만으로 계산할 수 있는 원소 특성의 통계값을 모델 입력으로 쓰는 표현.

소재 분야에서 이해하기: 구성 원소의 전기음성도 평균과 분산을 입력으로 쓴다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 용어집](https://scikit-learn.org/stable/glossary.html)

## 1. 조성만으로 기술자 만들기

원소 특성의 가중 평균·분산·최대최소 차이를 모아 입력을 만듭니다(Magpie 방식의 축소판).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

PROPERTIES = {   # (원자번호, 전기음성도, 원자반지름 A, 융점 K, 원자가전자수)
 'Li': (3, 0.98, 1.52, 454, 1), 'Be': (4, 1.57, 1.12, 1560, 2), 'Mg': (12, 1.31, 1.60, 923, 2),
 'Al': (13, 1.61, 1.43, 933, 3), 'Si': (14, 1.90, 1.11, 1687, 4), 'Ti': (22, 1.54, 1.47, 1941, 4),
 'Cr': (24, 1.66, 1.28, 2180, 6), 'Fe': (26, 1.83, 1.26, 1811, 8), 'Ni': (28, 1.91, 1.24, 1728, 10),
 'Cu': (29, 1.90, 1.28, 1358, 11),
}
NAMES = ['Z', 'electronegativity', 'radius', 'melting_point', 'valence']

def descriptors(composition):
    """composition: {원소: 분율}"""
    fractions = np.array(list(composition.values()), float)
    fractions = fractions / fractions.sum()
    matrix = np.array([PROPERTIES[element] for element in composition], float)
    mean = fractions @ matrix
    deviation = fractions @ np.abs(matrix - mean)
    return np.concatenate([mean, deviation, matrix.max(0) - matrix.min(0)])

FEATURE_NAMES = (['mean_' + name for name in NAMES] + ['dev_' + name for name in NAMES]
                 + ['range_' + name for name in NAMES])
example = descriptors({'Fe': 0.7, 'Cr': 0.2, 'Ni': 0.1})
for name, value in zip(FEATURE_NAMES, example):
    print('%-22s %.3f' % (name, value))

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

elements = list(PROPERTIES)
compositions, target = [], []
for _ in range(400):
    picked = rng.choice(elements, rng.integers(2, 5), replace=False)
    weights = rng.dirichlet(np.full(len(picked), 2.0))
    composition = dict(zip(picked, weights))
    vector = descriptors(composition)
    # 가상의 참 관계: 융점 평균과 원자반지름 편차가 지배
    value = 0.06 * vector[3] + 900 * vector[7] + rng.normal(0, 25)
    compositions.append(composition); target.append(value)

X_desc = np.array([descriptors(c) for c in compositions])
X_naive = np.zeros((len(compositions), len(elements)))
for row, composition in enumerate(compositions):
    for element, fraction in composition.items():
        X_naive[row, elements.index(element)] = fraction

target = np.array(target)
for name, features in [('원소 분율만 (one-hot)', X_naive), ('원소 특성 기술자', X_desc)]:
    score = cross_val_score(RandomForestRegressor(n_estimators=300, random_state=0),
                            features, target, cv=5, scoring='r2').mean()
    print('%-22s R2 %.3f' % (name, score))

## 2. 학습에 없던 원소가 나오면

In [ ]:
print('원소 분율 표현은 학습에 없던 원소의 열을 아예 갖지 못해 예측할 수 없습니다.')
print('원소 특성 기술자는 그 원소의 물성값만 있으면 벡터를 만들 수 있어 외삽이 가능합니다.')
print('  예: Zn 을 새로 추가 ->', np.round(descriptors({'Fe': 0.9, 'Cu': 0.1}), 2)[:5])
print('\n다만 학습 데이터가 덮지 않은 영역의 예측이라는 점은 그대로이므로 적용 범위 판정이 필요합니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#composition-descriptor)을 여세요.